# PCINet Attentive CutMix Training

**Based on:** `gradcam-pointing-games.ipynb`

**What is new:**
- Same-class Attentive CutMix augmentation during training
- Patch location guided by spatial attention from `model.features[-1]`
- Labels are unchanged (same-class mixing avoids ordinal label issues)
- Applied probabilistically per batch via `CUTMIX_PROB`
- Optional Test-time augmentation (TTA): original + horizontal flip averaged at inference

**Run scope:** trains the model, evaluates on the test set with TTA optionnaly, saves metrics.

In [2]:
import os, sys, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
import torchvision.transforms.functional as TF
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score, classification_report,
    accuracy_score, cohen_kappa_score, f1_score, top_k_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
import matplotlib.pyplot as plt
np.set_printoptions(threshold=sys.maxsize)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')


Using device: cuda:0
GPU: Tesla T4


In [ ]:
# Path to dataset
DATASET_DIR = '/kaggle/input/datasets/.../.../.../'
IMAGES_DIR  = f'{DATASET_DIR}/images'
TRAIN_CSV   = f'{DATASET_DIR}/train.csv'
VAL_CSV     = f'{DATASET_DIR}/val.csv'
TEST_CSV    = f'{DATASET_DIR}/test.csv'

# Path to the output directory
OUTPUT_DIR  = '/kaggle/working'

# Path to the best saved model in case retraining is not needed
SAVE_PATH = '/kaggle/input/datasets/.../.../.../best_model.pt'

# Path to image annotations
ANNOT_PATH  = '/kaggle/input/datasets/.../.../annotations.csv'

# Checking if the files needed from the dataset exist
for path, name in [(IMAGES_DIR,'IMAGES_DIR'),(TRAIN_CSV,'TRAIN_CSV'),(VAL_CSV,'VAL_CSV'),(TEST_CSV,'TEST_CSV')]:
    print(f'{"OK" if os.path.exists(path) else "NOT FOUND"}  {name}')

# Hyperparameters
EPOCHS       = 100
PATIENCE     = 15
BATCH_SIZE   = 16
LR           = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA  = 2
DROPOUT      = 0.4

# CutMix Parameters
CUTMIX_PROB  = 0.7
CUTMIX_ALPHA = 2.0

# Ordinal loss
ORDINAL_WEIGHT = 0.0

# TTA to average softmax over the original image and its horizontal flip at inference
# Can be disabled if wanted
TTA_ENABLED = True

# Set to False for retraining
SKIP_TRAINING = True

print('SKIP_TRAINING =', SKIP_TRAINING)
print('TTA_ENABLED   =', TTA_ENABLED)

## Dataset

In [ ]:
class MyRotationTransform:
    def __init__(self, angles):
        self.angles = angles
    def __call__(self, x):
        angle = np.random.choice(self.angles, p=[0.2, 0.6, 0.2])
        return TF.rotate(x, int(angle))


class CSVDataset(Dataset):
    def __init__(self, csv_path, images_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.images_dir, row['name'])).convert('RGB')
        label = int(row['label'])
        if self.transform: image = self.transform(image)
        return image, label

    def class_counts(self):
        return self.df['label'].value_counts().sort_index().tolist()


mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.1),
    MyRotationTransform(angles=[-90, 0, 90]),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 3.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transforms = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset = CSVDataset(TRAIN_CSV, IMAGES_DIR, transform=train_transforms)
val_dataset   = CSVDataset(VAL_CSV,   IMAGES_DIR, transform=eval_transforms)
test_dataset  = CSVDataset(TEST_CSV,  IMAGES_DIR, transform=eval_transforms)

#  Class weights via sklearn compute_class_weight to handle imbalanced dataset
train_labels_array = train_dataset.df['label'].values
CLASS_WEIGHTS = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2, 3, 4]),
    y=train_labels_array
).tolist()

#  WeightedRandomSampler using sklearn weights
sample_weights = np.array([CLASS_WEIGHTS[int(l)] for l in train_labels_array])
train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

counts = np.array(train_dataset.class_counts(), dtype=np.float32)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print('Train class counts (0-4):', counts.astype(int).tolist())
print('Class weights (sklearn): ', [f'{w:.3f}' for w in CLASS_WEIGHTS])

## Metrics

In [ ]:
# Defining the metrics
def calculate_metrics(preds, labels, scores, split='Val'):
    preds, labels = np.asarray(preds), np.asarray(labels)
    scores = np.asarray(scores)

    cm       = confusion_matrix(labels, preds, normalize='true')
    b_acc    = balanced_accuracy_score(labels, preds)
    acc      = accuracy_score(labels, preds)
    kappa    = cohen_kappa_score(labels, preds)
    f1       = f1_score(labels, preds, average='weighted')
    top2     = top_k_accuracy_score(labels, scores, k=2)
    top3     = top_k_accuracy_score(labels, scores, k=3)
    mae      = np.abs(preds - labels).mean()
    soft_acc = (np.abs(preds - labels) <= 1).mean()

    print(f'\n[{split}] Acc: {acc:.4f} | Bal Acc: {b_acc:.4f} | Kappa: {kappa:.4f} | '
          f'F1: {f1:.4f} | Top-2: {top2:.4f} | Top-3: {top3:.4f}')
    print(f'         MAE: {mae:.4f} | Soft Acc (±1): {soft_acc:.4f}  (paper: MAE=0.66, soft=0.8973)')
    print(cm)
    print(classification_report(labels, preds, zero_division=0))

    return {
        'accuracy': acc, 'balanced_accuracy': b_acc, 'kappa': kappa, 'f1': f1,
        'top_2_accuracy': top2, 'top_3_accuracy': top3,
        'mae': mae, 'soft_accuracy': soft_acc, 'confusion_matrix': cm
    }

## Focal Loss

In [ ]:
# Defining the loss function
class FocalLoss(nn.CrossEntropyLoss):
    def __init__(self, gamma, alpha=None, ignore_index=-100, reduction='none'):
        super().__init__(weight=alpha, ignore_index=ignore_index, reduction='none')
        self.reduction = reduction
        self.gamma = gamma

    def forward(self, input_, target):
        cross_entropy = super().forward(input_, target)
        target = target * (target != self.ignore_index).long()
        input_prob = torch.gather(F.softmax(input_, 1), 1, target.unsqueeze(1))
        loss = torch.pow(1 - input_prob, self.gamma) * cross_entropy
        if self.reduction == 'mean': return torch.mean(loss)
        elif self.reduction == 'sum': return torch.sum(loss)
        return loss

# The OFL
class OrdinalFocalLoss(nn.Module):
    """FocalLoss + ordinal MAE penalty on the expected predicted class.

    Penalises ordinal distance by computing the expected (mean) predicted
    class under the softmax distribution and adding L1 loss against the
    true label.  This directly minimises the MAE signal during training.
    """
    def __init__(self, gamma, alpha=None, ordinal_weight=0.1, num_classes=5):
        super().__init__()
        self.focal          = FocalLoss(gamma=gamma, alpha=alpha, reduction='mean')
        self.ordinal_weight = ordinal_weight
        self.register_buffer('class_idx', torch.arange(num_classes, dtype=torch.float))

    def forward(self, input_, target):
        focal    = self.focal(input_, target)
        probs    = F.softmax(input_, dim=1)
        expected = (probs * self.class_idx).sum(dim=1)
        mae      = F.l1_loss(expected, target.float())
        return focal + self.ordinal_weight * mae


## Model setup

In [ ]:
# Defining the model(Inherited from the original PCINet paper)
model = models.efficientnet_v2_s(weights='IMAGENET1K_V1')
num_ftrs = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT, inplace=True),
    nn.Linear(in_features=num_ftrs, out_features=5, bias=True),
)
model.to(device)

class_weights_tensor = torch.FloatTensor(CLASS_WEIGHTS).to(device)
criterion = OrdinalFocalLoss(gamma=FOCAL_GAMMA, alpha=class_weights_tensor,
                              ordinal_weight=ORDINAL_WEIGHT).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-8
)

print(model.classifier)
print(f'Dropout: {DROPOUT} | Weight decay: {WEIGHT_DECAY} | Early stopping patience: {PATIENCE}')
print(f'OrdinalFocalLoss: ordinal_weight={ORDINAL_WEIGHT}')
print('Checkpoint criterion: val MAE (minimise)')


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 193MB/s]


Sequential(
  (0): Dropout(p=0.4, inplace=True)
  (1): Linear(in_features=1280, out_features=5, bias=True)
)
Dropout: 0.4 | Weight decay: 0.0001 | Early stopping patience: 15
OrdinalFocalLoss: ordinal_weight=0.1
Checkpoint criterion: val MAE (minimise)


In [ ]:
# Attentive CutMix: same-class attention-guided patch mixing

# Run a no-grad forward pass and capture the spatial attention map from model.features[-1].
def get_feat_attn(model, images):
    maps = []
    def _hook(m, inp, out):
        maps.append(out.detach())
    handle = model.features[-1].register_forward_hook(_hook)
    with torch.no_grad():
        model(images)
    handle.remove()
    return maps[0].mean(dim=1)  # average across channels, one spatial map per image

# Same-class Attentive CutMix. For each image i in the batch, finds another
#     image j from the same PCI class, locates its attention peak, samples a patch
#     size from Beta(alpha, alpha), and pastes that patch from j into i.
def attentive_cutmix_batch(images, labels, attn_maps, alpha=1.0):
    B, C, H, W = images.shape
    Hf, Wf     = attn_maps.shape[1], attn_maps.shape[2]
    scale_y    = H / Hf
    scale_x    = W / Wf
    mixed      = images.clone()

    for i in range(B):
        # find another image from the same class
        same = (labels == labels[i]).nonzero(as_tuple=True)[0]
        same = same[same != i]
        if len(same) == 0:
            continue  # only one sample of this class in the batch

        j = same[random.randint(0, len(same) - 1)].item()

        # locate the attention peak of the source image
        attn_j = attn_maps[j]
        peak   = int(attn_j.argmax())
        fy, fx = peak // Wf, peak % Wf
        cy     = int(fy * scale_y)
        cx     = int(fx * scale_x)

        # sample patch size from a Beta distribution
        lam   = float(np.random.beta(alpha, alpha))
        cut_h = int(H * np.sqrt(1.0 - lam))
        cut_w = int(W * np.sqrt(1.0 - lam))

        # clamp the patch box to image boundaries
        y1 = max(0, cy - cut_h // 2)
        x1 = max(0, cx - cut_w // 2)
        y2 = min(H, y1 + cut_h)
        x2 = min(W, x1 + cut_w)

        mixed[i, :, y1:y2, x1:x2] = images[j, :, y1:y2, x1:x2]

    # Return mixed image with unchanged label
    return mixed  

print(f'  CUTMIX_PROB={CUTMIX_PROB}  CUTMIX_ALPHA={CUTMIX_ALPHA}')

## Training loop with early stopping

In [ ]:
if not SKIP_TRAINING:
    best_val_mae     = float('inf')
    no_improve_count = 0
    history = {
        'train_loss':[], 'val_loss':[],
        'train_bacc':[], 'val_bacc':[],
        'val_mae':[], 'val_soft_acc':[], 'lr':[]
    }
    metrics_history = []
    epoch_cms = []  # stores per-epoch confusion matrix
    val_support   = np.array(val_dataset.class_counts(), dtype=int)
    train_support = np.array(train_dataset.class_counts(), dtype=int)

    # Training Loop
    for ep in range(EPOCHS):
        print('=' * 55)
        print(f'Epoch {ep + 1}/{EPOCHS}  |  LR: {optimizer.param_groups[0]["lr"]:.2e}')

        # train
        model.train()
        losses, train_pred, train_score, train_labels = [], [], [], []
        for it, (inps, labs) in enumerate(train_loader):
            print(f'\r  Train {it+1}/{len(train_loader)}', end='', flush=True)
            inps, labs = inps.to(device), labs.to(device)

            # apply CutMix with probability CUTMIX_PROB
            if random.random() < CUTMIX_PROB:
                attn = get_feat_attn(model, inps)
                inps = attentive_cutmix_batch(inps, labs, attn, alpha=CUTMIX_ALPHA)

            optimizer.zero_grad()
            out  = model(inps)
            loss = criterion(out, labs)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            train_pred   += out.max(1)[1].cpu().numpy().tolist()
            train_score  += out.detach().cpu().numpy().tolist()
            train_labels += labs.cpu().numpy().tolist()

        train_loss = np.mean(losses)
        train_bacc = balanced_accuracy_score(train_labels, train_pred)
        train_cm   = confusion_matrix(train_labels, train_pred, normalize='true')
        history['train_loss'].append(train_loss)
        history['train_bacc'].append(train_bacc)
        print(f'\n  Train Loss: {train_loss:.4f} | Balanced Acc: {train_bacc:.4f}')

        # validate
        model.eval()
        losses, val_pred, val_score, val_labels = [], [], [], []
        with torch.no_grad():
            for it, (inps, labs) in enumerate(val_loader):
                print(f'\r  Val   {it+1}/{len(val_loader)}', end='', flush=True)
                inps, labs = inps.to(device), labs.to(device)
                out  = model(inps)
                loss = criterion(out, labs)
                losses.append(loss.item())
                val_pred   += out.max(1)[1].cpu().numpy().tolist()
                val_score  += out.detach().cpu().numpy().tolist()
                val_labels += labs.cpu().numpy().tolist()

        val_loss    = np.mean(losses)
        val_metrics = calculate_metrics(val_pred, val_labels, val_score, split='Val')
        val_bacc    = val_metrics['balanced_accuracy']
        val_mae     = val_metrics['mae']

        history['val_loss'].append(val_loss)
        history['val_bacc'].append(val_bacc)
        history['val_mae'].append(val_mae)
        history['val_soft_acc'].append(val_metrics['soft_accuracy'])
        history['lr'].append(optimizer.param_groups[0]['lr'])

        # Store confusion matrix
        epoch_cms.append({
            'epoch':    ep + 1,
            'train_cm': train_cm,
            'val_cm':   val_metrics['confusion_matrix'],
            'val_bacc': val_bacc,
            'val_mae':  val_mae,
        })

        epoch_record = {'epoch': ep+1, 'train_loss': train_loss, 'val_loss': val_loss,
                        'train_bacc': train_bacc}
        epoch_record.update({f'val_{k}': v for k, v in val_metrics.items()
                             if k != 'confusion_matrix'})
        metrics_history.append(epoch_record)

        # save if this is the best validation MAE so far
        scheduler.step(val_mae)

        if val_mae < best_val_mae:
            best_val_mae     = val_mae
            no_improve_count = 0
            torch.save(model.state_dict(), SAVE_PATH)
            print(f'  >>> New best val MAE: {best_val_mae:.4f} saved.')
        else:
            no_improve_count += 1
            print(f'  No improvement ({no_improve_count}/{PATIENCE})')

        # early stopping
        if no_improve_count >= PATIENCE:
            print(f'\nEarly stopping triggered at epoch {ep + 1}.')
            break

    print(f'\nTraining complete. Best val MAE: {best_val_mae:.4f}')
    pd.DataFrame(metrics_history).to_csv(f'{OUTPUT_DIR}/training_metrics.csv', index=False)
    print(' Training metrics saved')
else:
    print("Skipping training loading existing checkpoint:", SAVE_PATH)
    best_val_mae = None

## Test evaluation

In [ ]:
# Funtion for TTA
def evaluate_tta(model, loader, device):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for inps, labs in loader:
            inps      = inps.to(device)
            p_orig    = F.softmax(model(inps), dim=1)
            p_hflip   = F.softmax(model(torch.flip(inps, dims=[3])), dim=1)
            probs     = (p_orig + p_hflip) / 2
            all_probs.append(probs.cpu())
            all_labels.extend(labs.tolist())
    all_probs = torch.cat(all_probs, dim=0).numpy()
    preds     = all_probs.argmax(axis=1).tolist()
    return preds, all_probs.tolist(), all_labels

print('evaluate_tta() ready original + H-flip averaging')

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
model.eval()

# Evaluate with TTA when TTA_ENABLED is true
if TTA_ENABLED:
    print('Evaluating with TTA (original + H-flip)...')
    test_pred, test_score, test_labels = evaluate_tta(model, test_loader, device)
else:
    print('Evaluating without TTA (single pass)...')
    test_pred, test_score, test_labels = [], [], []
    with torch.no_grad():
        for inps, labs in test_loader:
            inps, labs = inps.to(device), labs.to(device)
            out = model(inps)
            test_pred   += out.max(1)[1].cpu().numpy().tolist()
            test_score  += out.detach().cpu().numpy().tolist()
            test_labels += labs.cpu().numpy().tolist()

test_metrics = calculate_metrics(test_pred, test_labels, test_score, split='Test')

pd.DataFrame([{k: v for k, v in test_metrics.items() if k != 'confusion_matrix'}]).to_csv(
    f'{OUTPUT_DIR}/test_metrics.csv', index=False)
print('\n Test metrics saved')

## Save predictions (with AE needed for GradCAM)

In [ ]:
test_df = pd.read_csv(TEST_CSV)
test_df['pred'] = test_pred
test_df['ae']   = np.abs(np.array(test_pred) - np.array(test_labels))
for c in range(5):
    test_df[f'score_{c}'] = [s[c] for s in test_score]

test_df.to_csv(f'{OUTPUT_DIR}/test_predictions.csv', index=False)
print(' test_predictions.csv saved')

print('\nAE group breakdown:')
for ae_val, label in [(0,'AE=0 (correct)'),(1,'AE=1 (near-miss)')]:
    n = (test_df['ae'] == ae_val).sum()
    print(f'  {label}: {n} ({100*n/len(test_df):.1f}%)')
n = (test_df['ae'] > 1).sum()
print(f'  AE>1 (harsh error): {n} ({100*n/len(test_df):.1f}%)')

✓ test_predictions.csv saved

AE group breakdown:
  AE=0 (correct): 375 (46.5%)
  AE=1 (near-miss): 361 (44.8%)
  AE>1 (harsh error): 70 (8.7%)


## Save confusion matrices (training + test)

In [ ]:
if not SKIP_TRAINING:
    # Creating a readable confusion matrix
    def format_cm(norm_cm, support, title):
        lines = []
        lines.append(title)
        lines.append('-' * len(title))

        # Header
        header = f'  {"True\\Pred":<12}' + ''.join(f'  {"PCI " + str(j):<10}' for j in range(5))
        lines.append(header)
        lines.append('  ' + '-' * (len(header) - 2))

        for i in range(5):
            row_support = support[i] if support is not None else None
            row_str = f'  PCI {i}'
            if row_support is not None:
                row_str += f' (n={row_support:<4})'
            else:
                row_str = f'  PCI {i:<10}'
            for j in range(5):
                pct   = norm_cm[i][j] * 100
                count = round(norm_cm[i][j] * support[i]) if support is not None else ''
                cell  = f'{pct:.0f}% ({count})' if support is not None else f'{pct:.0f}%'
                row_str += f'  {cell:<10}'
            lines.append(row_str)

        return '\n'.join(lines)


    test_support = np.array([
        (np.array(test_labels) == c).sum() for c in range(5)
    ], dtype=int)

    out_path = f'{OUTPUT_DIR}/confusion_matrices.txt'
    with open(out_path, 'w') as f:

        f.write('=' * 70 + '\n')
        f.write('CONFUSION MATRICES PCINet Training Run\n')
        f.write('Each cell: % of true class predicted as that column class (count)\n')
        f.write('Diagonal cells = correct predictions\n')
        f.write('=' * 70 + '\n\n')

        #  Per epoch training and validation matrix 
        for rec in epoch_cms:
            ep = rec['epoch']
            marker = ' *** BEST CHECKPOINT ***' if rec['val_mae'] == best_val_mae else ''
            f.write(f'{"=" * 70}\n')
            f.write(f'EPOCH {ep}  |  Val Balanced Acc: {rec["val_bacc"]:.4f}  |  Val MAE: {rec["val_mae"]:.4f}{marker}\n')
            f.write(f'{"=" * 70}\n\n')

            f.write(format_cm(rec['train_cm'], train_support, 'Training Set CM'))
            f.write('\n\n')
            f.write(format_cm(rec['val_cm'], val_support, 'Validation Set CM'))
            f.write('\n\n')

        #  Test Confusion Matrix 
        f.write('=' * 70 + '\n')
        f.write('FINAL TEST SET CONFUSION MATRIX\n')
        f.write(f'Acc: {test_metrics["accuracy"]:.4f}  |  Bal Acc: {test_metrics["balanced_accuracy"]:.4f}  |  '
                f'MAE: {test_metrics["mae"]:.4f}  |  Soft Acc: {test_metrics["soft_accuracy"]:.4f}\n')
        f.write('=' * 70 + '\n\n')
        f.write(format_cm(test_metrics['confusion_matrix'], test_support, 'Test Set CM'))
        f.write('\n')

    print(f' confusion_matrices.txt saved to {out_path}')
    print(f'  {len(epoch_cms)} training epochs + 1 test run')
else:
    print("Confusion matrices skipped (SKIP_TRAINING=True). Already saved from previous run.")


✓ confusion_matrices.txt saved to /kaggle/working/confusion_matrices.txt
  55 training epochs + 1 test run


## Training curves

In [ ]:
# Creating training curves to make monitoring the training easier
if not SKIP_TRAINING:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    eps = range(1, len(history['train_loss']) + 1)

    axes[0].plot(eps, history['train_loss'], label='train', color='#378ADD')
    axes[0].plot(eps, history['val_loss'],   label='val',   color='#1D9E75')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    _best_bacc = max(history['val_bacc'])
    axes[1].plot(eps, history['train_bacc'], label='train', color='#378ADD')
    axes[1].plot(eps, history['val_bacc'],   label='val',   color='#1D9E75')
    axes[1].axhline(_best_bacc, color='red', linestyle='--', alpha=0.5,
                    label=f'best={_best_bacc:.3f}')
    axes[1].set_title('Balanced Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(eps, history['val_mae'],      label='MAE',      color='#D85A30')
    axes[2].plot(eps, history['val_soft_acc'], label='Soft acc', color='#534AB7')
    axes[2].axhline(0.66,   color='#D85A30', linestyle='--', alpha=0.4, label='paper MAE 0.66')
    axes[2].axhline(0.8973, color='#534AB7', linestyle='--', alpha=0.4, label='paper soft 0.897')
    axes[2].set_title('MAE & Soft Accuracy vs paper'); axes[2].set_xlabel('Epoch')
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/training_curves.pdf', bbox_inches='tight')
    plt.show()
    print(' Training curves saved')
else:
    print("Training curves skipped (SKIP_TRAINING=True). Already saved from previous run.")

## Generate Visualization Images

In [ ]:
# VISUALIZATION A: original | CutMix augmented (with patch box)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch.nn.functional as F
import zipfile, os

model.eval()

N_VIZ = 20
all_orig, all_mixed, all_boxes, all_lbls = [], [], [], []
all_attn_maps, H_viz, W_viz = [], None, None

viz_dataset = CSVDataset(TRAIN_CSV, IMAGES_DIR, transform=eval_transforms)
viz_loader = DataLoader(viz_dataset, batch_size=16, shuffle=True, num_workers=0)
mn_t  = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(device)
std_t = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(device)
def denorm_v(t): return (t * std_t + mn_t).clamp(0,1).cpu()

for imgs_v, lbls_v in viz_loader:
    imgs_v = imgs_v.to(device)
    lbls_v = lbls_v.to(device)
    attn_v = get_feat_attn(model, imgs_v)
    B_v, C_v, H_v, W_v = imgs_v.shape
    Hf_v, Wf_v = attn_v.shape[1], attn_v.shape[2]
    if H_viz is None: H_viz, W_viz = H_v, W_v

    mixed_v = imgs_v.clone()
    boxes_v = [None] * B_v
    for i in range(B_v):
        same = (lbls_v == lbls_v[i]).nonzero(as_tuple=True)[0]
        same = same[same != i]
        if len(same) == 0:
            continue
        j      = same[random.randint(0, len(same)-1)].item()
        peak   = int(attn_v[j].argmax())
        fy, fx = peak // Wf_v, peak % Wf_v
        cy, cx = int(fy * H_v / Hf_v), int(fx * W_v / Wf_v)
        lam    = float(np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA))
        cut_h  = int(H_v * np.sqrt(1.0 - lam))
        cut_w  = int(W_v * np.sqrt(1.0 - lam))
        y1 = max(0, cy - cut_h // 2); x1 = max(0, cx - cut_w // 2)
        y2 = min(H_v, y1 + cut_h);    x2 = min(W_v, x1 + cut_w)
        mixed_v[i, :, y1:y2, x1:x2] = imgs_v[j, :, y1:y2, x1:x2]
        boxes_v[i] = (y1, x1, y2, x2)

    for k in range(B_v):
        if boxes_v[k] is None:
            continue
        all_orig.append(denorm_v(imgs_v[k:k+1])[0].permute(1,2,0).numpy())
        all_mixed.append(denorm_v(mixed_v[k:k+1])[0].permute(1,2,0).numpy())
        all_attn_maps.append(attn_v[k].cpu())
        all_boxes.append(boxes_v[k])
        all_lbls.append(lbls_v[k].item())
        if len(all_orig) >= N_VIZ:
            break
    if len(all_orig) >= N_VIZ:
        break

print(f'Collected {len(all_orig)} examples')

# Save individual files
out_dir_a = f'{OUTPUT_DIR}/cutmix_A_examples'
os.makedirs(out_dir_a, exist_ok=True)

for idx in range(len(all_orig)):
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
    fig.suptitle(f'PCI {all_lbls[idx]}   Attentive CutMix',
                 fontsize=11, fontweight='bold')
    axes[0].imshow(all_orig[idx]);  axes[0].axis('off'); axes[0].set_title('Original', fontsize=10)
    axes[1].imshow(all_mixed[idx]); axes[1].axis('off'); axes[1].set_title('After CutMix  (red = pasted patch)', fontsize=10)
    y1, x1, y2, x2 = all_boxes[idx]
    rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2.5, edgecolor='red', facecolor='none')
    axes[1].add_patch(rect)
    plt.tight_layout()
    plt.savefig(f'{out_dir_a}/cutmix_A_{idx+1:02d}_pci{all_lbls[idx]}.png',
                dpi=150, bbox_inches='tight')
    plt.close()

# Zip
zip_path = f'{OUTPUT_DIR}/cutmix_A_examples.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(out_dir_a)):
        zf.write(os.path.join(out_dir_a, fname), fname)
print(f'Saved {len(all_orig)} individual PNGs -> cutmix_A_examples.zip')


In [ ]:
# VISUALIZATION B : original | attention map | CutMix augmented
import torch.nn.functional as F
import zipfile, os

out_dir_b = f'{OUTPUT_DIR}/cutmix_B_examples'
os.makedirs(out_dir_b, exist_ok=True)

# Upsample and normalise attention maps
attn_stack = torch.stack(all_attn_maps, dim=0)  # (N, Hf, Wf)
attn_up = F.interpolate(
    attn_stack.unsqueeze(1).float(), size=(H_viz, W_viz),
    mode='bilinear', align_corners=False
).squeeze(1)  # (N, H, W)
mn_a = attn_up.flatten(1).min(1)[0].view(-1,1,1)
mx_a = attn_up.flatten(1).max(1)[0].view(-1,1,1)
attn_norm = (attn_up - mn_a) / (mx_a - mn_a + 1e-8)

for idx in range(len(all_orig)):
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    fig.suptitle(f'PCI {all_lbls[idx]}   Attentive CutMix: how the patch location is chosen',
                 fontsize=11, fontweight='bold')
    axes[0].imshow(all_orig[idx]);  axes[0].axis('off'); axes[0].set_title('Original', fontsize=10)
    axes[1].imshow(all_orig[idx])
    axes[1].imshow(attn_norm[idx].numpy(), cmap='jet', alpha=0.55)
    axes[1].axis('off'); axes[1].set_title('Attention map  (patch centred on peak)', fontsize=10)
    axes[2].imshow(all_mixed[idx]); axes[2].axis('off'); axes[2].set_title('After CutMix', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{out_dir_b}/cutmix_B_{idx+1:02d}_pci{all_lbls[idx]}.png',
                dpi=150, bbox_inches='tight')
    plt.close()

zip_path = f'{OUTPUT_DIR}/cutmix_B_examples.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(out_dir_b)):
        zf.write(os.path.join(out_dir_b, fname), fname)
print(f'Saved {len(all_orig)} individual PNGs -> cutmix_B_examples.zip')
